# Feynman diagram: $e^+e^- \to \mu^+\mu^-$ from Dirac traces

*nuclear · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/nuclear/feynman_ee_mumu)

The simplest QED diagram: an electron and a positron annihilate into a virtual photon that makes a muon pair. The Feynman rules turn the picture into an amplitude, the spin sums turn it into traces of $4\times4$ Dirac matrices $\gamma^\mu$, and each trace is one einsum. Sweeping the energy shows the angular shape change from isotropic at threshold to $1 + \cos^2\theta$, and the cross section fall as $4\pi\alpha^2/3s$.

**Vector form**

$$\mathcal M = \frac{e^2}{s}\,\big[\bar v(p')\,\gamma^\mu\,u(p)\big]\big[\bar u(k)\,\gamma_\mu\,v(k')\big],\qquad \frac{d\sigma}{d\Omega} = \frac{1}{64\pi^2 s}\,\frac{|\mathbf k|}{|\mathbf p|}\,\overline{|\mathcal M|^2}$$

**Index (tensor) form**

$$\overline{|\mathcal M|^2} = \frac{e^4}{4s^2}\,L^{\mu\nu}M_{\mu\nu},\qquad L^{\mu\nu} = (\not p' - m_e)_{ab}\,\gamma^\mu_{bc}\,(\not p + m_e)_{cd}\,\gamma^\nu_{da},\qquad M_{\mu\nu} = (\not k + m)_{ab}\,\gamma_{\mu,bc}\,(\not k' - m)_{cd}\,\gamma_{\nu,da}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\gamma^\mu_{ab}$ | `gamma_matrices(device)` | (4, 4, 4) | Dirac matrices: index $\mu$ is spacetime, $a, b$ are spinor indices; $\{\gamma^\mu, \gamma^\nu\} = 2\eta^{\mu\nu}$ |
| $\eta_{\mu\nu}$ | `metric(device)` | (4, 4) | Minkowski metric diag(1, −1, −1, −1), units with $\hbar = c = 1$ |
| $p^\mu, p'^\mu, k^\mu, k'^\mu$ | `momenta(sqrt_s, theta, m)` | (E, B, 4) | centre-of-mass four-momenta of $e^-$, $e^+$, $\ell^-$, $\ell^+$ for every energy $\sqrt s$ and angle $\theta$ |
| $\not p_{ab} = \gamma^\mu_{ab}\,\eta_{\mu\nu}\,p^\nu$ | `einsum('mab,mn,...n->...ab', gamma, eta, p)` | (E, B, 4, 4) | Feynman slash: the spin sums $\sum u\bar u = \not p + m$ and $\sum v\bar v = \not p - m$ |
| $L^{\mu\nu}$ | `einsum('...ab,mbc,...cd,nda->...mn', A, gamma, B, gamma)` | (E, B, 4, 4) | repeated $a, b, c, d$ ⇒ summed: the trace of the electron line (same function for the muon line) |
| $L^{\mu\nu}M_{\mu\nu}$ | `einsum('...mn,mr,ns,...rs->...', L, eta, eta, M)` | (E, B) | lower the muon indices with $\eta$ and contract: the photon propagator joins the two lines |
| $d\sigma/d\Omega$ | `dsigma_domega(msq, sqrt_s, m_e, m)` | (E, B) | flux and phase-space factors; $(\hbar c)^2$ converts GeV$^{-2}$ to nb |
| $\sigma = 2\pi\int \frac{d\sigma}{d\Omega}\sin\theta\,d\theta$ | `trapezoid(dsdo * sin(theta), theta)` | (E,) | total cross section; compared with $\frac{4\pi\alpha^2}{3s}\beta\,(1 + 2m^2/s)$, $\beta = \sqrt{1 - 4m^2/s}$ |

The diagram is drawn once, but the code never builds a spinor: summing over spins replaces $u\bar u$ by $\not p + m$, so the whole calculation is products and traces of $\gamma$ matrices, which einsum does for every energy and angle at once. The result is $\overline{|\mathcal M|^2} = e^4\big[1 + 4m^2/s + (1 - 4m^2/s)\cos^2\theta\big]$ for $m_e \to 0$: at threshold the slow muons come out isotropically, far above it the photon's spin 1 and helicity conservation give $1 + \cos^2\theta$. The total $\sigma$ rises steeply from $\sqrt s = 2m$ and then falls as $1/s$, about $86.8$ nb at 1 GeV. Real data from PETRA and LEP follow this curve until $\sqrt s \approx 30$ GeV, where a second diagram with a $Z$ boson instead of the photon interferes and makes the forward–backward asymmetry $A_{FB}$ non-zero; pure QED gives $A_{FB} = 0$. Try the $\tau$ pair, discovered at SPEAR in 1975 just above its 3.55 GeV threshold.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/nuclear/feynman_ee_mumu/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

import torch

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import sigma_point_like, trace_tensor


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    title, m_f, sym = tensors.FINAL[params["final"]]
    gamma, eta = tensors.gamma_matrices(dev), tensors.metric(dev)  # (4, 4, 4), (4, 4)

    # Clifford algebra check: {γ^μ, γ^ν} = 2η^{μν}, so Tr(γ^μ γ^ν) = 4η^{μν}.
    one = torch.eye(4, dtype=gamma.dtype, device=dev)
    tr2 = trace_tensor(one, one, gamma).real
    print(f"Tr(γ^μ γ^ν) = 4η^μν: max error {(tr2 - 4 * eta.real).abs().max().item():.1e}")

    threshold = 2 * m_f
    sqrt_s_max = max(params["sqrt_s_max"], threshold * 1.5)
    theta = tensors.angles(91, dev)  # (B,)
    sqrt_s = tensors.energies(threshold, sqrt_s_max, 120, dev)  # (E,)
    dsdo, sigma = solve.cross_sections(sqrt_s, theta, m_f, tensors.ALPHA, gamma, eta)  # (E, B), (E,)
    exact = sigma_point_like(sqrt_s, m_f, tensors.ALPHA)
    afb = solve.forward_backward(dsdo, theta)

    print(f"{title}: threshold √s = 2m_{sym} = {threshold:.4f} GeV, sweep up to {sqrt_s_max:.1f} GeV")
    rel = ((sigma - exact) / exact).abs().max()
    print(f"σ from the traces vs 4πα²β(1+2m²/s)/3s: max relative difference {rel:.1e}")
    for j in (len(sqrt_s) // 6, len(sqrt_s) // 2, len(sqrt_s) - 1):
        shape = dsdo[j, 0] / dsdo[j, len(theta) // 2]
        print(
            f"√s = {sqrt_s[j]:7.3f} GeV   σ = {sigma[j]:9.3f} nb   "
            f"dσ/dΩ(0°)/dσ/dΩ(90°) = {shape:.3f}   A_FB = {afb[j]:+.1e}"
        )
    print("high-energy limit: dσ/dΩ ∝ 1 + cos²θ, so the 0°/90° ratio → 2; σ·s → 4πα²/3 = 86.8 nb·GeV²")
    return plot.animate(sqrt_s, theta, dsdo, sigma, exact, (title, sym, threshold), out_dir / "feynman_ee_mumu.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/nuclear/feynman_ee_mumu/equation.py
r"""Physics → tensor translation.

One Feynman diagram: e⁻(p) e⁺(p') annihilate into a virtual photon, q = p + p', which makes ℓ⁻(k) ℓ⁺(k').
The Feynman rules (vertex −ieγ^μ, propagator −iη_{μν}/q², spinors u, v) give

    \mathcal M = \frac{e^2}{s}\,[\bar v(p')\gamma^\mu u(p)]\,[\bar u(k)\gamma_\mu v(k')],     s = q^2.

Summing |M|² over all spins turns the spinors into traces of 4×4 Dirac matrices (Σ u ū = p̸ + m, Σ v v̄ = p̸ − m,
with p̸ = γ^μ η_{μν} p^ν), and averaging over the 4 initial spin states:

    \overline{|\mathcal M|^2} = \frac{e^4}{4 s^2}\,L^{\mu\nu} M_{\mu\nu},
    L^{\mu\nu} = \mathrm{Tr}[(\not p' - m_e)\gamma^\mu(\not p + m_e)\gamma^\nu],
    M_{\mu\nu} = \mathrm{Tr}[(\not k + m)\gamma_\mu(\not k' - m)\gamma_\nu].

Each trace is one einsum over the spinor indices a, b, c, d, and L^{μν}M_{μν} is one more over μ, ν. In the
centre-of-mass frame the cross section per solid angle is

    \frac{d\sigma}{d\Omega} = \frac{1}{64\pi^2 s}\,\frac{|\mathbf k|}{|\mathbf p|}\,\overline{|\mathcal M|^2}.
"""

import math

import torch

GEV2_TO_NB = 0.3893794e6  # (ħc)² in GeV² nb: converts GeV⁻² to nanobarn


def slash(p: torch.Tensor, gamma: torch.Tensor, eta: torch.Tensor) -> torch.Tensor:
    r"""p̸_{ab} = γ^μ_{ab} η_{μν} p^ν   (μ, ν summed).   p: (..., 4) real → (..., 4, 4) complex"""
    return torch.einsum("mab,mn,...n->...ab", gamma, eta, p.to(gamma.dtype))


def trace_tensor(a: torch.Tensor, b: torch.Tensor, gamma: torch.Tensor) -> torch.Tensor:
    r"""T^{μν} = Tr[A γ^μ B γ^ν] = A_{ab} γ^μ_{bc} B_{cd} γ^ν_{da}   (a, b, c, d summed).
    a, b: (..., 4, 4) → (..., 4, 4)"""
    return torch.einsum("...ab,mbc,...cd,nda->...mn", a, gamma, b, gamma)


def amplitude_squared(p, pb, k, kb, m_e: float, m_f: float, alpha: float, gamma, eta) -> torch.Tensor:
    r"""Spin-averaged |M|² = e⁴/(4s²) L^{μν} η_{μρ} η_{νσ} M^{ρσ}.   momenta: (..., 4) → (...)"""
    one = torch.eye(4, dtype=gamma.dtype, device=gamma.device)
    lepton = trace_tensor(slash(pb, gamma, eta) - m_e * one, slash(p, gamma, eta) + m_e * one, gamma)  # L^{μν}
    muon = trace_tensor(slash(k, gamma, eta) + m_f * one, slash(kb, gamma, eta) - m_f * one, gamma)  # M^{ρσ}
    contracted = torch.einsum("...mn,mr,ns,...rs->...", lepton, eta, eta, muon)  # L^{μν} M_{μν}
    q = (p + pb).to(gamma.dtype)
    s = torch.einsum("...m,mn,...n->...", q, eta, q).real
    e2 = 4 * math.pi * alpha
    return (e2**2 / (4 * s**2) * contracted).real


def dsigma_domega(msq: torch.Tensor, sqrt_s: torch.Tensor, m_e: float, m_f: float) -> torch.Tensor:
    r"""dσ/dΩ = |k|/|p| · |M|²/(64π² s) in nb/sr.   msq: (E, B), sqrt_s: (E,) → (E, B)"""
    s = sqrt_s**2
    ratio = ((s / 4 - m_f**2).clamp(min=0) / (s / 4 - m_e**2)).sqrt()
    return (ratio / (64 * math.pi**2 * s))[:, None] * msq * GEV2_TO_NB


def sigma_point_like(sqrt_s: torch.Tensor, m_f: float, alpha: float) -> torch.Tensor:
    r"""σ = (4πα²/3s) β (1 + 2m²/s), β = √(1 − 4m²/s), m_e → 0, in nb.   (E,) → (E,)"""
    s = sqrt_s**2
    x = 4 * m_f**2 / s
    beta = (1 - x).clamp(min=0).sqrt()
    return 4 * math.pi * alpha**2 / (3 * s) * beta * (1 + x / 2) * GEV2_TO_NB


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/nuclear/feynman_ee_mumu/tensors.py
"""Build the concrete tensors: Dirac matrices, the metric, lepton masses and the four momenta of each event."""

import math

import torch

ALPHA = 1 / 137.035999  # fine-structure constant, e² = 4πα
M_E = 0.000511  # electron mass, GeV

# final state: (label, mass in GeV, symbol)
FINAL = {
    "muon": ("e⁺e⁻ → μ⁺μ⁻", 0.105658, "μ"),
    "tau": ("e⁺e⁻ → τ⁺τ⁻", 1.77686, "τ"),
}


def metric(device: torch.device) -> torch.Tensor:
    """η_{μν} = diag(1, −1, −1, −1): shape (4, 4)."""
    return torch.diag(torch.tensor([1.0, -1.0, -1.0, -1.0], dtype=torch.complex128, device=device))


def gamma_matrices(device: torch.device) -> torch.Tensor:
    """Dirac representation γ^μ_{ab}: γ^0 = diag(1, 1, −1, −1), γ^i = [[0, σ_i], [−σ_i, 0]]: shape (4, 4, 4)."""
    sigma = torch.tensor(
        [[[0, 1], [1, 0]], [[0, -1j], [1j, 0]], [[1, 0], [0, -1]]], dtype=torch.complex128, device=device
    )  # (3, 2, 2) Pauli matrices
    eye = torch.eye(2, dtype=torch.complex128, device=device)
    zero = torch.zeros(2, 2, dtype=torch.complex128, device=device)
    g = torch.zeros(4, 4, 4, dtype=torch.complex128, device=device)
    g[0] = torch.block_diag(eye, -eye)
    for i in range(3):
        g[i + 1] = torch.cat([torch.cat([zero, sigma[i]], 1), torch.cat([-sigma[i], zero], 1)], 0)
    return g


def angles(n: int, device: torch.device) -> torch.Tensor:
    """Scattering angles θ of the outgoing lepton relative to the incoming electron, 0 … π: shape (B,)."""
    return torch.linspace(0.0, math.pi, n, dtype=torch.float64, device=device)


def energies(threshold: float, sqrt_s_max: float, n: int, device: torch.device) -> torch.Tensor:
    """Centre-of-mass energies √s, log-spaced from just above threshold to sqrt_s_max: shape (E,)."""
    lo = math.log(threshold * 1.002)
    return torch.logspace(lo / math.log(10), math.log10(sqrt_s_max), n, dtype=torch.float64, device=device)


def momenta(sqrt_s: torch.Tensor, theta: torch.Tensor, m_f: float) -> tuple[torch.Tensor, ...]:
    """Centre-of-mass four-momenta p (e⁻), p' (e⁺), k (ℓ⁻), k' (ℓ⁺): four tensors of shape (E, B, 4).

    Beams along ±z; the outgoing pair back to back in the x–z plane at angle θ to the electron.
    """
    e = (sqrt_s / 2)[:, None].expand(-1, theta.shape[0])  # (E, B) beam energy
    p_in = (e**2 - M_E**2).sqrt()
    p_out = (e**2 - m_f**2).clamp(min=0).sqrt()
    zero = torch.zeros_like(e)
    s, c = torch.sin(theta).expand_as(e), torch.cos(theta).expand_as(e)
    p = torch.stack([e, zero, zero, p_in], -1)
    pb = torch.stack([e, zero, zero, -p_in], -1)
    k = torch.stack([e, p_out * s, zero, p_out * c], -1)
    kb = torch.stack([e, -p_out * s, zero, -p_out * c], -1)
    return p, pb, k, kb


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/nuclear/feynman_ee_mumu/solve.py
"""Evaluate the diagram on an (energy × angle) grid and integrate it into the total cross section."""

import math

import torch

from problems._common.progress import steps

from .equation import amplitude_squared, dsigma_domega
from .tensors import M_E, momenta


def cross_sections(sqrt_s, theta, m_f: float, alpha: float, gamma, eta, chunk: int = 16):
    """dσ/dΩ (E, B) in nb/sr and σ (E,) in nb = 2π ∫ dσ/dΩ sin θ dθ (trapezoid rule)."""
    rows = []
    for i in steps(math.ceil(sqrt_s.shape[0] / chunk), "energy chunks"):
        e = sqrt_s[i * chunk : (i + 1) * chunk]
        p, pb, k, kb = momenta(e, theta, m_f)  # (e, B, 4) each
        msq = amplitude_squared(p, pb, k, kb, M_E, m_f, alpha, gamma, eta)  # (e, B)
        rows.append(dsigma_domega(msq, e, M_E, m_f))
    dsdo = torch.cat(rows)
    sigma = 2 * math.pi * torch.trapezoid(dsdo * torch.sin(theta), theta, dim=-1)
    return dsdo, sigma


def forward_backward(dsdo: torch.Tensor, theta: torch.Tensor) -> torch.Tensor:
    """A_FB = (σ_F − σ_B)/(σ_F + σ_B): zero for pure QED (the 1 + cos²θ shape is symmetric). (E, B) → (E,)"""
    w = dsdo * torch.sin(theta)
    mid = theta.shape[0] // 2  # θ grid has an odd number of points, so theta[mid] = 90° closes both halves
    f = torch.trapezoid(w[..., : mid + 1], theta[: mid + 1], dim=-1)
    b = torch.trapezoid(w[..., mid:], theta[mid:], dim=-1)
    return (f - b) / (f + b)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/nuclear/feynman_ee_mumu/plot.py
"""Sweep the collision energy: angular distribution as a polar shape, total cross section against √s."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

CB, CO, CG = "#2563eb", "#f59e0b", "#9ca3af"


def animate(
    sqrt_s: torch.Tensor,
    theta: torch.Tensor,
    dsdo: torch.Tensor,
    sigma: torch.Tensor,
    exact: torch.Tensor,
    labels: tuple[str, str, float],
    path: Path,
) -> Path:
    """dsdo (E, B) nb/sr, sigma and exact (E,) nb."""
    title, sym, threshold = labels
    e, th = sqrt_s.cpu().numpy(), theta.cpu().numpy()
    shape = dsdo.cpu().numpy()
    shape = shape / shape[:, len(th) // 2 : len(th) // 2 + 1]  # normalise to 1 at θ = 90°
    sig, ex = sigma.cpu().numpy(), exact.cpu().numpy()
    full = np.concatenate([th, 2 * np.pi - th[::-1]])  # mirror: the distribution is symmetric about the beam

    fig, (ax_p, ax_s) = plt.subplots(1, 2, figsize=(9.5, 4.2), gridspec_kw={"width_ratios": [1, 1.25]})
    ax_p.set_xlim(-2.3, 2.3)
    ax_p.set_ylim(-2.3, 2.3)
    ax_p.set_aspect("equal")
    ax_p.axhline(0, color="#374151", lw=0.8)
    ax_p.annotate("", xy=(2.2, 0), xytext=(1.4, 0), arrowprops={"arrowstyle": "->", "color": "#374151"})
    ax_p.text(1.45, 0.12, "e⁻ beam", fontsize=8)
    ref = 1 + np.cos(full) ** 2
    ax_p.plot(ref * np.cos(full), ref * np.sin(full), ls="--", color=CO, lw=1.5, label="1 + cos²θ  (√s ≫ 2m)")
    ax_p.plot(np.cos(full), np.sin(full), ls=":", color=CG, lw=1.2, label="isotropic  (threshold)")
    (curve,) = ax_p.plot([], [], color=CB, lw=2.2, label="traces")
    ax_p.legend(loc="lower center", fontsize=7, frameon=False, ncols=1, bbox_to_anchor=(0.5, -0.02))
    ax_p.set_title(f"dσ/dΩ shape for {sym}⁺{sym}⁻ (1 at 90°)", fontsize=10)
    ax_p.set_xticks([])
    ax_p.set_yticks([])
    readout = ax_p.text(0.02, 0.97, "", transform=ax_p.transAxes, va="top", fontsize=9, family="monospace")

    ax_s.loglog(e, ex, color=CO, lw=3, alpha=0.5, label="4πα²β(1+2m²/s)/3s")
    ax_s.loglog(e, 86.86 / e**2, color=CG, ls=":", lw=1.2, label="4πα²/3s  (massless)")
    ax_s.loglog(e, sig, color=CB, lw=1.5, label="∫ traces dΩ")
    ax_s.axvline(threshold, color="#dc2626", lw=0.8, ls="--")
    ax_s.text(threshold * 1.05, sig.max() * 0.4, f"2m_{sym}", color="#dc2626", fontsize=8)
    (dot,) = ax_s.plot([], [], "o", color=CB, ms=7)
    ax_s.set_xlabel("√s (GeV)")
    ax_s.set_ylabel("σ (nb)")
    ax_s.set_title(title, fontsize=10)
    ax_s.legend(fontsize=7, frameon=False)
    ax_s.grid(True, which="both", alpha=0.2)

    def update(j: int):
        r = np.concatenate([shape[j], shape[j][::-1]])
        curve.set_data(r * np.cos(full), r * np.sin(full))
        dot.set_data([e[j]], [sig[j]])
        readout.set_text(f"√s = {e[j]:.3f} GeV\nσ  = {sig[j]:.3g} nb")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(e), 90), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'final': 'muon',
    'sqrt_s_max': 10.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.nuclear.feynman_ee_mumu")]:
    del sys.modules[name]
main = importlib.import_module("problems.nuclear.feynman_ee_mumu.main")
problem_dir = Path("problems/nuclear/feynman_ee_mumu")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
